# Prefill-Time Amnesic — Full Self-Contained Notebook

**Hypothesis (Yap 2603.16335)**: On Qwen3.5-35B-A3B (our sibling architecture), behavioral commitments are computed during **PREFILL** in GatedDeltaNet state. Our current amnesic at the last prompt token (+2.7pp) may be too late.

**Test**: apply amnesic ablation at multiple positions during prefill. 8 configs, N=112, ~5-8 min compute.

**Self-contained**: loads model, fits probes, runs sweep, analyzes. No prior session state required.

**Prerequisites on Colab**:
- RTX 6000 Blackwell (96 GB) runtime
- `HF_TOKEN` secret set (has access to Qwen3.6 + mcr-stage-b)

**Verdicts automatizados no Cell 8**:
- ⭐⭐⭐ > +5pp Pareto → Yap confirmed in reasoning
- ⭐⭐ +3-5pp → moderate confirmation
- ⭐ +1.5-3pp → prefill comparable to last-token
- ⚖️ no advantage → Yap is behavior-specific, not reasoning

## Cell 1 — Install stack (~5 min first time, 30s cached)

In [ ]:
import sys, subprocess, os, shutil
from pathlib import Path

def pip(*a, check=True):
    return subprocess.run([sys.executable, '-m', 'pip', *a], check=check)

try:
    import transformers
    from transformers.models.auto.configuration_auto import CONFIG_MAPPING_NAMES
    ok = 'qwen3_5_moe' in CONFIG_MAPPING_NAMES
except Exception:
    ok = False

if not ok:
    pip('install', '-q', 'accelerate', 'datasets', 'huggingface_hub==1.5.0',
        'safetensors', 'einops', 'scikit-learn', 'sentencepiece', 'tokenizers',
        'protobuf', 'scipy', 'matplotlib', 'hf_transfer', check=False)
    pip('uninstall', '-y', 'transformers', check=False)
    SRC = '/content/transformers_src'
    if Path(SRC).exists(): shutil.rmtree(SRC)
    subprocess.run(['git','clone','--quiet','--depth=1',
                    'https://github.com/huggingface/transformers.git', SRC], check=True)
    pip('install','--force-reinstall','--no-deps','--no-cache-dir', SRC, check=False)
    pip('install','--no-cache-dir','flash-linear-attention', check=False)
    pip('install','-q','--no-cache-dir',
        'https://github.com/Dao-AILab/causal-conv1d/releases/download/v1.6.1.post4/'
        'causal_conv1d-1.6.1%2Bcu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl',
        check=False)
    for m in list(sys.modules):
        if m.startswith('transformers') or m.startswith('huggingface_hub'):
            del sys.modules[m]

try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
    if hf_token:
        from huggingface_hub import login
        login(token=hf_token)
        print('HF auth OK')
except Exception as e:
    print(f'(skipping colab auth: {e})')

import json, re, pickle, time, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'

OUT = Path('/content/prefill_amnesic'); OUT.mkdir(exist_ok=True)
print('setup done')

## Cell 2 — Load Qwen3.6-35B-A3B (~4 min)

In [ ]:
from transformers import AutoTokenizer, AutoModelForImageTextToText
from huggingface_hub import snapshot_download
from safetensors import safe_open

MODEL_ID = 'Qwen/Qwen3.6-35B-A3B'
LAYERS = [11, 17]  # L11 reference + L17 target

tok = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tok.pad_token_id is None: tok.pad_token = tok.eos_token

model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID, dtype=torch.bfloat16, device_map='cuda',
    attn_implementation='sdpa', trust_remote_code=True)
model.eval()
print(f'Model loaded: {torch.cuda.memory_allocated()/1e9:.1f} GB')

## Cell 3 — Fit probes at L11, L17 + build letter directions (~3 min)

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression

corpus = snapshot_download('caiovicentino1/Qwen3.6-35B-A3B-mcr-stage-b',
                            repo_type='dataset', cache_dir='/content/cache')
shards = sorted((Path(corpus)/'shards').glob('q*.safetensors'))
print(f'{len(shards)} shards')

MIN_LEN = 200; POOL_WINDOW = 10
pooled_by_layer = {L: [] for L in LAYERS}
labels = []

for shard in tqdm(shards, desc='collect'):
    with safe_open(str(shard), framework='pt') as f:
        meta = f.metadata()
        offs_all = json.loads(meta['offsets'])
        rollouts = json.loads(meta['rollouts'])
        acts_dict = {L: f.get_tensor(f'acts_L{L}') for L in LAYERS}
        for r_idx, r in enumerate(rollouts):
            if r['pred'] is None or r['response_len'] < MIN_LEN: continue
            for L in LAYERS:
                offs = offs_all[f'L{L}']
                ra = acts_dict[L][offs[r_idx]:offs[r_idx+1]].float().numpy()
                pooled_by_layer[L].append(ra[:POOL_WINDOW].mean(axis=0))
            labels.append(r['pred'])

labels = np.array(labels)
letters = sorted(set(labels))
letter_to_int = {l: i for i, l in enumerate(letters)}
y = np.array([letter_to_int[l] for l in labels])
print(f'n={len(labels)}, letters={letters}')

d_stacks = {}
for L in LAYERS:
    X = np.stack(pooled_by_layer[L])
    scaler = StandardScaler().fit(X)
    pca = PCA(n_components=128, random_state=42).fit(scaler.transform(X))
    lr = LogisticRegression(C=1.0, max_iter=3000, random_state=42).fit(
        pca.transform(scaler.transform(X)), y)
    print(f'L{L} train: {lr.score(pca.transform(scaler.transform(X)), y):.3f}')
    dirs = []
    for li in range(len(letters)):
        coef = lr.coef_[li]
        d_scaled = pca.components_.T @ coef
        d_raw = d_scaled * scaler.scale_
        d_raw = d_raw / (np.linalg.norm(d_raw) + 1e-8)
        dirs.append(torch.from_numpy(d_raw.astype(np.float32)).cuda().to(torch.bfloat16))
    d_stacks[L] = torch.stack(dirs)  # [10, d_model]

print(f'\n✅ d_stacks for layers {LAYERS}')

## Cell 4 — Prefill-aware AmnesicHook + install

In [ ]:
def get_layer_module(m, idx):
    cands = [m]
    if hasattr(m, 'model'): cands.append(m.model)
    for start in cands:
        for path in [('model','language_model','layers'),('language_model','layers'),('model','layers'),('layers',)]:
            cur = start; ok = True
            for p in path:
                if hasattr(cur, p): cur = getattr(cur, p)
                else: ok = False; break
            if ok and hasattr(cur, '__getitem__'):
                try: return cur[idx]
                except: continue
    raise RuntimeError(f'layer {idx} not found')

class AmnesicHookPrefill:
    """Apply ablation at multiple positions during prefill (T == prompt_len).
    positions_mode: 'last' | 'all_prefill' | 'first_half' | 'last_half' | 'every_N'
    """
    def __init__(self, layer_idx):
        self.layer_idx = layer_idx
        self.mode = 'off'
        self.positions_mode = 'last'
        self.prompt_len = None
        self.alpha = 1.0
        self.every_n = 1

    def set(self, mode, positions_mode, prompt_len, alpha=1.0, every_n=1):
        self.mode = mode
        self.positions_mode = positions_mode
        self.prompt_len = prompt_len
        self.alpha = alpha
        self.every_n = every_n

    def off(self):
        self.mode = 'off'

    def make_hook(self):
        def hook(module, inp, out):
            if self.mode == 'off': return out
            hidden = out[0] if isinstance(out, tuple) else out
            T = hidden.shape[1]
            if T != self.prompt_len: return out  # only during prefill
            if self.positions_mode == 'last':
                positions = [T - 1]
            elif self.positions_mode == 'all_prefill':
                positions = list(range(T))
            elif self.positions_mode == 'first_half':
                positions = list(range(T // 2))
            elif self.positions_mode == 'last_half':
                positions = list(range(T // 2, T))
            elif self.positions_mode == 'every_N':
                positions = list(range(0, T, self.every_n))
            else:
                return out
            hidden = hidden.clone()
            ds = d_stacks[self.layer_idx]
            x_all = hidden[0]
            coefs = x_all[positions] @ ds.T
            delta = coefs @ ds
            hidden[0, positions, :] = x_all[positions] - self.alpha * delta
            if isinstance(out, tuple): return (hidden,) + out[1:]
            return hidden
        return hook

hooks = {L: AmnesicHookPrefill(L) for L in LAYERS}
handles = {L: get_layer_module(model, L).register_forward_hook(hooks[L].make_hook()) for L in LAYERS}
print(f'✅ prefill-aware hooks on {LAYERS}')

## Cell 5 — Load eval questions (Stage B 10-opt, ≥1 correct rollout)

In [ ]:
questions = []
for shard in shards:
    with safe_open(str(shard), framework='pt') as f:
        meta = f.metadata()
        rollouts_ = json.loads(meta['rollouts'])
        opts = json.loads(meta['options'])
        n_correct = sum(1 for r in rollouts_ if r['correct'])
        if len(opts) == 10 and n_correct >= 1:
            questions.append(dict(
                question=meta['question'], options=opts,
                gold_letter=meta['gold_letter'], n_options=10, n_correct=n_correct))
print(f'{len(questions)} Stage B 10-opt questions (≥1 correct)')

def format_mcq(question, options):
    choices = '\n'.join(f'{chr(65+i)}. {opt}' for i, opt in enumerate(options))
    content = ("Answer the following multiple-choice question. "
        "Give ONLY the letter of the correct answer.\n\n"
        f"Question: {question}\n\nOptions:\n{choices}\n\nAnswer: \\boxed{{")
    return tok.apply_chat_template([{'role':'user','content':content}],
                                    tokenize=False, add_generation_prompt=True,
                                    enable_thinking=False)

letter_ids = {L: tok(L, add_special_tokens=False).input_ids[0] for L in 'ABCDEFGHIJ'}
print('letter_ids:', letter_ids)

## Cell 6 — Configs

In [ ]:
# (name, mode, positions_mode, alpha, every_n, target_layer)
CONFIGS_PREFILL = [
    ('baseline',           'off',  'last',         1.0, 1,  17),
    ('L17_last',           'all',  'last',         1.0, 1,  17),  # current winner ref
    ('L17_all_prefill',    'all',  'all_prefill',  1.0, 1,  17),  # Yap hypothesis
    ('L17_first_half',     'all',  'first_half',   1.0, 1,  17),  # early prefill
    ('L17_last_half',      'all',  'last_half',    1.0, 1,  17),  # late prefill
    ('L17_every_4',        'all',  'every_N',      1.0, 4,  17),  # sparse 1/4
    ('L17_every_8',        'all',  'every_N',      1.0, 8,  17),  # sparser 1/8
    ('L11_all_prefill',    'all',  'all_prefill',  1.0, 1,  11),  # L11 prefill-wide
]

def set_prefill(mode, pos_mode, target_layer, prompt_len, alpha, every_n):
    for L in LAYERS: hooks[L].off()
    if mode == 'off': return
    hooks[target_layer].set(mode, pos_mode, prompt_len, alpha, every_n)

print(f'{len(CONFIGS_PREFILL)} configs')

## Cell 7 — Sweep (~5-8 min)

In [ ]:
random.seed(42)
sample = random.sample(questions, min(200, len(questions)))
print(f'{len(sample)} questions × {len(CONFIGS_PREFILL)} configs')

results_prefill = []
t0 = time.time()
for i, q in enumerate(tqdm(sample, desc='prefill sweep')):
    try:
        p = format_mcq(q['question'], q['options'])
        ids = tok(p, return_tensors='pt').input_ids.to('cuda')
        if ids.shape[1] > 4096: continue
        prompt_len = ids.shape[1]
        row = dict(idx=i, gold=q['gold_letter'], prompt_len=prompt_len)
        for (name, mode, pos_mode, alpha, every_n, target_layer) in CONFIGS_PREFILL:
            set_prefill(mode, pos_mode, target_layer, prompt_len, alpha, every_n)
            with torch.no_grad():
                out = model(input_ids=ids, attention_mask=torch.ones_like(ids))
            logits = out.logits[0, -1]
            letter_logits = {L: logits[letter_ids[L]].item() for L in 'ABCDEFGHIJ'[:q['n_options']]}
            pred = max(letter_logits, key=letter_logits.get)
            row[f'{name}_pred'] = pred
            row[f'{name}_correct'] = (pred == q['gold_letter'])
        for L in LAYERS: hooks[L].off()
        results_prefill.append(row)
        if (i+1) % 20 == 0:
            accs = {name: sum(r[f'{name}_correct'] for r in results_prefill)/len(results_prefill) for (name,*_) in CONFIGS_PREFILL}
            top = sorted(accs.items(), key=lambda x: -x[1])[:4]
            print(f'  [{i+1}/{len(sample)}] top4: ' + ' | '.join(f'{n}={a*100:.0f}%' for n,a in top))
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache(); continue

with open(OUT/'prefill_results.json', 'w') as f:
    json.dump(dict(n=len(results_prefill), configs=[c[0] for c in CONFIGS_PREFILL],
                   results=results_prefill, wall_min=(time.time()-t0)/60), f, indent=2)
print(f'\n✅ {(time.time()-t0)/60:.1f} min')

## Cell 8 — Analysis + verdict

In [ ]:
from scipy.stats import binomtest

print(f'=== Prefill Sweep (N={len(results_prefill)}) ===\n')
bc = [r['baseline_correct'] for r in results_prefill]
ba = sum(bc) / len(results_prefill) * 100
print(f'baseline              : {ba:.1f}%\n')
header_delta = 'Delta_pp'
print(f'{"config":22s}  {"acc%":>6s}  {header_delta:>9s}  {"gain":>5s}  {"lost":>5s}  {"p":>7s}')
stats = []
for (name, *_) in CONFIGS_PREFILL:
    if name == 'baseline': continue
    cc = [r[f'{name}_correct'] for r in results_prefill]
    acc = sum(cc) / len(results_prefill)
    delta = (acc - ba/100) * 100
    g = sum(1 for b, c in zip(bc, cc) if not b and c)
    l = sum(1 for b, c in zip(bc, cc) if b and not c)
    p = binomtest(g, g + l, p=0.5, alternative='two-sided').pvalue if g + l > 0 else 1.0
    stats.append((name, acc * 100, delta, g, l, p))

stats.sort(key=lambda r: -r[2])
for name, acc, delta, g, l, p in stats:
    if delta > 5 and l == 0: star = ' ***'
    elif delta > 3 and l == 0: star = ' **'
    elif delta > 1.5 and l == 0: star = ' *'
    elif delta > 0.5 and l == 0: star = ' +'
    else: star = ''
    print(f'{name:22s}: {acc:5.1f}%  {delta:+8.1f}   {g:>5d}  {l:>5d}  {p:>7.3f}{star}')

best = max(stats, key=lambda r: r[2])
best_pareto = max([s for s in stats if s[4] == 0], key=lambda s: s[2], default=None)

print('\n=== VERDICT ===')
if best_pareto and best_pareto[2] > 5:
    print(f'*** BREAKTHROUGH: {best_pareto[0]} -> {best_pareto[2]:+.1f}pp Pareto')
    print('    Yap 2603.16335 CONFIRMED on reasoning. Commitment is prefill-time in Qwen3.6.')
elif best_pareto and best_pareto[2] > 3:
    print(f'** STRONG: {best_pareto[0]} -> {best_pareto[2]:+.1f}pp Pareto')
    print('   Prefill-wide ablation beats last-token. Moderate Yap confirmation.')
elif best_pareto and best_pareto[2] > 1.5:
    print(f'* Moderate: {best_pareto[0]} -> {best_pareto[2]:+.1f}pp, 0 lost')
    print('   Prefill comparable to last-token. No dramatic improvement.')
else:
    print(f'-- No prefill advantage: best {best[0]} {best[2]:+.1f}pp')
    print('   Yap finding may be behavior-specific, not reasoning. Last-token remains optimal.')

last_delta = next((s[2] for s in stats if s[0] == 'L17_last'), 0)
prefill_delta = next((s[2] for s in stats if s[0] == 'L17_all_prefill'), 0)
print(f'\n--- Core comparison ---')
print(f'L17 last-token:    {last_delta:+.1f}pp')
print(f'L17 all_prefill:   {prefill_delta:+.1f}pp')
print(f'Prefill advantage: {prefill_delta - last_delta:+.1f}pp')